# Ordered Logistic Regression Results (FAIR^2 Dataset) Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Do not subscript or iterate as dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list the available record sets, then drill down into their fields and columns using their `@id`s.

> **Note:** All references to dataset elements use their `@id` for consistency.

In [ ]:
# List all record sets by @id and get their info
print("Available record sets (@id, name):")
record_sets = list(dataset.record_sets)
record_set_ids = []
for rs in record_sets:
    print(f"  {rs['@id']} — {rs.get('name', '[no name]')}")
    record_set_ids.append(rs['@id'])

# For each record set, list field @ids and column @ids if available
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']} — {rs.get('name', '[no name]')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    if len(fields) > 0:
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"    {f['@id']} — {f.get('name', '[no name]')}")
            elif isinstance(f, str):
                print(f"    {f}")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if len(columns) > 0:
        print("  Columns:")
        for c in columns:
            if isinstance(c, dict):
                print(f"    {c['@id']} — {c.get('name', '[no name]')}")
            elif isinstance(c, str):
                print(f"    {c}")

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame. 
All record sets are referenced via their `@id`.
If you need to explore a specific record set in detail, set its `@id` in the cells below.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
print('Attempting to load all available record sets into DataFrames:')
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded: {record_set_id} — shape: {df.shape}")
    except Exception as e:
        print(f"  Could not load {record_set_id}: {e}")

# Preview the columns of the first loaded DataFrame (if available)
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"\nColumns in {first_rs_id}:\n", dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No record sets loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing values, and grouping data. 
All columns/fields referenced here use their `@id` from the dataset schema.

> **Tip:** Adjust the `record_set_id`, `numeric_field_id`, and `group_field_id` variables below based on your overview.

In [ ]:
# --- Configuration: Change these to relevant @ids from your record set ---
# Set these after inspecting output from data overview above, e.g.: 
# record_set_id = 'cr:SummaryStatistics' (example, not real unless seen in overview)
# numeric_field_id = 'cr:log_likelihood' (example)

# For illustration, we'll auto-detect numeric columns in the first DataFrame loaded, if any.
import numpy as np

if dataframes:
    record_set_id = first_rs_id
    df = dataframes[record_set_id]
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # Try to pick the first available numeric field
    if num_cols:
        numeric_field_id = num_cols[0]
        print(f'Using record set: {record_set_id}, numeric field: {numeric_field_id}')
    else:
        print("No numeric fields found in the first record set.")
        numeric_field_id = None

    # Filtering: Only proceed if numeric_field_id is set
    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean()  # Choose mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        norm_colname = f"{numeric_field_id}_normalized"
        filtered_df[norm_colname] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_colname]].head())

        # Try grouping by another column (prefer categorical columns)
        # Attempt to find a likely group field (object type with few unique values)
        cat_cols = [col for col in df.columns if df[col].dtype == 'object' and df[col].nunique() > 1 and df[col].nunique() < len(df)//2]
        if cat_cols:
            group_field_id = cat_cols[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"Grouped data by {group_field_id} (showing mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize data distributions and relationships.

You can adjust the field and grouping below based on your data. This example creates a histogram of the selected numeric field, and a boxplot grouped by the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field_id exists, show boxplot
    if 'group_field_id' in locals() and group_field_id is not None:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the dataset using the Croissant schema with `mlcroissant`, explored available record sets and their fields (referenced by `@id`), and performed initial data analysis and visualization.

Remember: all entity references in this notebook use the `@id` to ensure clarity and reproducibility for any future data schema updates.

> For further exploration, customize the fields, record sets, and analysis steps using the `@id` values you discovered in Section 2. Consider extending this notebook with additional visualizations or statistical summaries.